# HFCS selected raw data: diagnostic audit

**Purpose:** audit the 50 selected-raw Parquet extracts created by `load_hfcs.ipynb` before any cleaning or variable construction. No source file is changed and no household-level rows or values are exported.

- Inputs: sibling `HFCS Data Download/HFCS_selected_raw/hfcs_<wave>_<h|d>_imp<1–5>.parquet`.
- Outputs: `Research-Assistant-Git/HFCS/diagnostics/` (CSV files, directly readable in Excel).
- The five imputations are **not** treated as independent households. Diagnostics retain imputation-level results, with a separate descriptive summary across imputations.
- Original Stata extended missing values (`.a`, `.b`, etc.) are examined through the `__stata_missing` companion columns created by the earlier export cell. Nulls and extended missing codes are reported separately.
- Negative numeric codes are **flagged**, not reinterpreted as missing. Categorical code frequencies use imputation 1 only and suppress potentially identifying small cells.

**Limitation:** the selected Parquet exports contain numeric codes but not necessarily the *Stata value-label dictionaries*. Their meaning must be checked against the original `.dta` files or official HFCS codebooks. This notebook can inspect code distributions, not certify the correct label mapping.

## 1. Configuration

In [1]:
from __future__ import annotations

import os
import re
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Expected survey years and imputations, as used in the export cell.
WAVES = (2010, 2014, 2017, 2021, 2023)
IMPUTATIONS = (1, 2, 3, 4, 5)
TABLES = ("h", "d")

# Tune these to control the volume of categorical-code diagnostics.
MAX_CATEGORY_CODES = 12
MIN_FREQUENCY_TO_PUBLISH = 5
CATEGORICAL_PREFIXES = ("hh", "dh")

# Optional manual override; otherwise locate the repository automatically.
REPO_ROOT_OVERRIDE = None


def discover_repo() -> Path:
    explicit = REPO_ROOT_OVERRIDE or os.environ.get("HFCS_REPO_ROOT")
    if explicit:
        root = Path(explicit).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(root)
        return root
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "hfcs_file_inventory.txt").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError(
        "Open Jupyter from Research-Assistant-Git or set REPO_ROOT_OVERRIDE."
    )


REPO = discover_repo()
INPUT = REPO.parent / "HFCS Data Download" / "HFCS_selected_raw"
OUTPUT = REPO / "HFCS" / "diagnostics"

if not INPUT.is_dir():
    raise FileNotFoundError(f"Selected raw Parquet folder not found: {INPUT}")

OUTPUT.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO)
print("Reading from:", INPUT)
print("Writing aggregates to:", OUTPUT)

Repository: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\Research-Assistant-Git
Reading from: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download\HFCS_selected_raw
Writing aggregates to: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\Research-Assistant-Git\HFCS\diagnostics


## 2. File inventory and column metadata

This checks that all 50 expected files exist. It also records the number of rows and the names/types of exported variables, without reading the data values.

In [2]:
def parquet_path(wave: int, table: str, imp: int) -> Path:
    return INPUT / f"hfcs_{wave}_{table}_imp{imp}.parquet"


def clean_name_map(names):
    result = {}
    for name in names:
        lower = name.lower()
        if lower in result and name != result[lower]:
            raise ValueError(f"Case-insensitive duplicate columns: {name}")
        result[lower] = name
    return result


file_records = []
schema_records = []
issues = []

for wave in WAVES:
    for imp in IMPUTATIONS:
        for table in TABLES:
            path = parquet_path(wave, table, imp)
            record = {
                "wave": wave, "table": table, "imputation": imp,
                "filename": path.name, "exists": path.is_file(),
                "rows": pd.NA, "raw_variables": pd.NA,
                "missing_code_companion_columns": pd.NA,
                "size_mb": round(path.stat().st_size / (1024**2), 2) if path.exists() else pd.NA,
            }
            if path.exists():
                meta = pq.ParquetFile(path)
                names = meta.schema_arrow.names
                real_names = [c for c in names if not c.endswith("__stata_missing")]
                record["rows"] = meta.metadata.num_rows
                record["raw_variables"] = len(real_names)
                record["missing_code_companion_columns"] = len(names) - len(real_names)
                for f in meta.schema_arrow:
                    schema_records.append({
                        "wave": wave, "table": table, "imputation": imp,
                        "variable": f.name,
                        "parquet_type": str(f.type),
                        "stata_missing_companion": f.name.endswith("__stata_missing"),
                    })
                del meta
            else:
                issues.append({"wave": wave, "country": "ALL", "table": table,
                               "imputation": imp, "variable": "", "issue": "missing_parquet_file",
                               "detail": path.name})
            file_records.append(record)

files_df = pd.DataFrame(file_records)
schema_df = pd.DataFrame(schema_records)
display(files_df.groupby(["wave", "table"], as_index=False)["exists"].agg(["sum", "count"]))
print(f"Found {int(files_df.exists.sum())} / {len(files_df)} expected files")
if not files_df.exists.any():
    raise FileNotFoundError("No expected Parquet files found. Check INPUT path.")

,wave,table,sum,count
0,2010,d,5,5
1,2010,h,5,5
2,2014,d,5,5
3,2014,h,5,5
4,2017,d,5,5
5,2017,h,5,5
6,2021,d,5,5
7,2021,h,5,5
8,2023,d,5,5
9,2023,h,5,5


Found 50 / 50 expected files


## 3. Inspect countries, missingness, codes and identifiers

Read **one Parquet file at a time**. No row is dropped or changed in the source files. We inspect every variable for missingness. Code-frequency tables are deliberately restricted to discrete numeric-coded questionnaire variables, with imputation 1 used for display.

In [3]:
sample_rows = []
missing_rows = []
code_rows = []
extended_rows = []
text_rows = []


def country_groups(df: pd.DataFrame, country_col: str):
    # Converted to strings only for diagnostic grouping; source values remain intact.
    labels = df[country_col].astype("string").fillna("<MISSING_COUNTRY>")
    return df.groupby(labels, sort=False, dropna=False)


def is_categorical_candidate(name: str, series: pd.Series) -> bool:
    if not name.lower().startswith(CATEGORICAL_PREFIXES):
        return False
    # Age and other continuous demographic quantities are not survey response codes.
    if name.lower().startswith(("dhage", "dhbirth")):
        return False
    if not pd.api.types.is_numeric_dtype(series):
        return False
    values = series.dropna()
    if values.empty:
        return False
    distinct = values.unique()
    if len(distinct) > MAX_CATEGORY_CODES:
        return False
    try:
        arr = np.asarray(distinct, dtype=float)
    except (TypeError, ValueError):
        return False
    return bool(np.isfinite(arr).all() and np.isclose(arr, np.rint(arr)).all())


def one_group(wave: int, table: str, imp: int, country: str,
              sub: pd.DataFrame, real_cols: list[str], lookup: dict[str, str]):
    total = len(sub)
    household_key = lookup.get("sa0010")
    missing_key_rows = int(sub[household_key].isna().sum()) if household_key else pd.NA
    duplicate_key_rows = int(sub[household_key].duplicated(keep=False).sum()) if household_key else pd.NA
    sample_rows.append({
        "wave": wave, "country": country, "table": table, "imputation": imp,
        "rows": total, "missing_household_id_rows": missing_key_rows,
        "duplicate_household_id_rows": duplicate_key_rows,
        "distinct_household_ids": int(sub[household_key].nunique(dropna=True)) if household_key else pd.NA,
        "raw_variables": len(real_cols),
    })
    if household_key is None:
        issues.append({"wave": wave, "country": country, "table": table,
                       "imputation": imp, "variable": "sa0010",
                       "issue": "missing_household_key", "detail": "Identifier not exported"})
    elif missing_key_rows or duplicate_key_rows:
        issues.append({"wave": wave, "country": country, "table": table,
                       "imputation": imp, "variable": "sa0010",
                       "issue": "household_id_problem",
                       "detail": f"missing={missing_key_rows}, duplicate_rows={duplicate_key_rows}"})

    for col in real_cols:
        s = sub[col]
        companion = lookup.get((col + "__stata_missing").lower())
        nulls = s.isna()
        extended = sub[companion].notna() if companion else pd.Series(False, index=sub.index)
        extended_n = int(extended.sum())
        normal_missing_n = int((nulls & ~extended).sum())
        overlap_n = int((~nulls & extended).sum())
        missing_n = int((nulls | extended).sum())
        nonmissing_n = total - missing_n
        blank_strings = 0
        replacement_chars = 0
        suspicious_text_chars = 0
        if pd.api.types.is_string_dtype(s) or s.dtype == "object":
            strings = s.dropna()
            if not strings.empty and (pd.api.types.is_string_dtype(s) or isinstance(strings.iloc[0], str)):
                try:
                    strings = strings.astype("string")
                    blank_strings = int(strings.str.fullmatch(r"\s*", na=False).sum())
                    replacement_chars = int(strings.str.contains("\ufffd", regex=False, na=False).sum())
                    suspicious_text_chars = int(strings.str.contains(r"[\u0080-\u009f]", regex=True, na=False).sum())
                except (TypeError, ValueError):
                    pass
        missing_rows.append({
            "wave": wave, "country": country, "table": table, "imputation": imp,
            "variable": col, "dtype": str(s.dtype), "rows": total,
            "observed_n": nonmissing_n, "missing_total_n": missing_n,
            "missing_pct": round(100 * missing_n / total, 4) if total else pd.NA,
            "ordinary_null_n": normal_missing_n,
            "stata_extended_missing_n": extended_n,
            "empty_string_n": blank_strings,
            "unique_observed_n": int(s.nunique(dropna=True)),
        })
        if overlap_n:
            issues.append({"wave": wave, "country": country, "table": table,
                           "imputation": imp, "variable": col,
                           "issue": "stata_companion_non_null_but_value_present",
                           "detail": f"{overlap_n} rows"})
        if replacement_chars or suspicious_text_chars:
            text_rows.append({"wave": wave, "country": country, "table": table,
                              "imputation": imp, "variable": col,
                              "unicode_replacement_character_rows": replacement_chars,
                              "unicode_control_character_rows": suspicious_text_chars})
        if companion and extended_n:
            code_counts = sub.loc[extended, companion].astype("string").value_counts(dropna=False)
            for code, count in code_counts.items():
                extended_rows.append({
                    "wave": wave, "country": country, "table": table, "imputation": imp,
                    "variable": col, "stata_missing_code": str(code), "count": int(count),
                })
        if imp == 1 and is_categorical_candidate(col, s) and total >= 50:
            value_counts = s.value_counts(dropna=True)
            # Suppress entire variable distribution when any category has <5 rows.
            # This avoids publishing a tiny category frequency in a public repo.
            if not value_counts.empty and (value_counts < MIN_FREQUENCY_TO_PUBLISH).any():
                continue
            for value, count in value_counts.items():
                code_rows.append({
                    "wave": wave, "country": country, "table": table,
                    "variable": col, "raw_numeric_code": str(value),
                    "count": int(count),
                    "pct_of_nonmissing": round(100 * count / nonmissing_n, 3)
                    if nonmissing_n else pd.NA,
                })


for record in file_records:
    if not record["exists"]:
        continue
    wave, table, imp = record["wave"], record["table"], record["imputation"]
    path = parquet_path(wave, table, imp)
    print(f"Inspecting {path.name} ...", flush=True)
    df = pd.read_parquet(path)
    lookup = clean_name_map(df.columns)
    real_cols = [c for c in df.columns if not c.endswith("__stata_missing")]
    country_col = lookup.get("sa0100")
    if country_col is None:
        issues.append({"wave": wave, "country": "ALL", "table": table,
                       "imputation": imp, "variable": "sa0100",
                       "issue": "missing_country_key", "detail": "Cannot produce country groups"})
    else:
        if "im0100" in lookup:
            codes = df[lookup["im0100"]].dropna()
            if not codes.empty and not codes.astype("string").str.replace(r"\.0$", "", regex=True).eq(str(imp)).all():
                issues.append({"wave": wave, "country": "ALL", "table": table,
                               "imputation": imp, "variable": "im0100",
                               "issue": "imputation_code_mismatch", "detail": path.name})
        for country, sub in country_groups(df, country_col):
            one_group(wave, table, imp, str(country), sub, real_cols, lookup)
    del df
    gc.collect()

samples_df = pd.DataFrame(sample_rows)
missing_df = pd.DataFrame(missing_rows)
print("Country-by-file rows:", len(samples_df), "Variable-level checks:", len(missing_df))
if not samples_df.empty:
    display(samples_df.head(12))

Inspecting hfcs_2010_h_imp1.parquet ...
Inspecting hfcs_2010_d_imp1.parquet ...
Inspecting hfcs_2010_h_imp2.parquet ...
Inspecting hfcs_2010_d_imp2.parquet ...
Inspecting hfcs_2010_h_imp3.parquet ...
Inspecting hfcs_2010_d_imp3.parquet ...
Inspecting hfcs_2010_h_imp4.parquet ...
Inspecting hfcs_2010_d_imp4.parquet ...
Inspecting hfcs_2010_h_imp5.parquet ...
Inspecting hfcs_2010_d_imp5.parquet ...
Inspecting hfcs_2014_h_imp1.parquet ...
Inspecting hfcs_2014_d_imp1.parquet ...
Inspecting hfcs_2014_h_imp2.parquet ...
Inspecting hfcs_2014_d_imp2.parquet ...
Inspecting hfcs_2014_h_imp3.parquet ...
Inspecting hfcs_2014_d_imp3.parquet ...
Inspecting hfcs_2014_h_imp4.parquet ...
Inspecting hfcs_2014_d_imp4.parquet ...
Inspecting hfcs_2014_h_imp5.parquet ...
Inspecting hfcs_2014_d_imp5.parquet ...
Inspecting hfcs_2017_h_imp1.parquet ...
Inspecting hfcs_2017_d_imp1.parquet ...
Inspecting hfcs_2017_h_imp2.parquet ...
Inspecting hfcs_2017_d_imp2.parquet ...
Inspecting hfcs_2017_h_imp3.parquet ...


,wave,country,table,imputation,rows,missing_household_id_rows,duplicate_household_id_rows,distinct_household_ids,raw_variables
0,2010,AT,h,1,2380,0,0,2380,23
1,2010,BE,h,1,2327,0,0,2327,23
2,2010,CY,h,1,1237,0,0,1237,23
3,2010,DE,h,1,3565,0,0,3565,23
4,2010,E1,h,1,6197,0,0,6197,23
5,2010,ES,h,1,6106,0,0,6106,23
6,2010,FI,h,1,10989,0,0,10989,23
7,2010,FR,h,1,15006,0,0,15006,23
8,2010,GR,h,1,2971,0,0,2971,23
9,2010,IT,h,1,7951,0,0,7951,23


## 4. Check whether H and D have matching household keys

This reads **identifier columns only**, one wave and imputation at a time. No merge is saved and no household record leaves the computer.

In [4]:
join_rows = []
KEYS = ("sa0100", "sa0010", "im0100")

for wave in WAVES:
    for imp in IMPUTATIONS:
        hp, dp = parquet_path(wave, "h", imp), parquet_path(wave, "d", imp)
        if not hp.exists() or not dp.exists():
            continue
        h_schema = clean_name_map(pq.ParquetFile(hp).schema_arrow.names)
        d_schema = clean_name_map(pq.ParquetFile(dp).schema_arrow.names)
        keys = [k for k in KEYS if k in h_schema and k in d_schema]
        if not {"sa0100", "sa0010"}.issubset(keys):
            issues.append({"wave": wave, "country": "ALL", "table": "h+d",
                           "imputation": imp, "variable": "keys",
                           "issue": "cannot_match_h_d", "detail": f"shared keys: {keys}"})
            continue
        h = pd.read_parquet(hp, columns=[h_schema[k] for k in keys])
        d = pd.read_parquet(dp, columns=[d_schema[k] for k in keys])
        h.columns = keys
        d.columns = keys
        h_dupes, d_dupes = int(h.duplicated(keys).sum()), int(d.duplicated(keys).sum())
        if h_dupes or d_dupes:
            issues.append({"wave": wave, "country": "ALL", "table": "h+d",
                           "imputation": imp, "variable": "keys",
                           "issue": "duplicate_h_d_merge_keys",
                           "detail": f"h={h_dupes}, d={d_dupes}"})
        # Distinct-key comparison, without multiplying matches from duplicates.
        merged = h.drop_duplicates().merge(d.drop_duplicates(), how="outer", on=keys,
                                            indicator=True, validate="one_to_one")
        for country, sub in country_groups(merged, "sa0100"):
            status = sub["_merge"].value_counts()
            only_h = int(status.get("left_only", 0))
            only_d = int(status.get("right_only", 0))
            both = int(status.get("both", 0))
            join_rows.append({"wave": wave, "country": str(country), "imputation": imp,
                              "matched_keys": both, "h_only_keys": only_h,
                              "d_only_keys": only_d})
            if only_h or only_d:
                issues.append({"wave": wave, "country": str(country), "table": "h+d",
                               "imputation": imp, "variable": "keys",
                               "issue": "h_d_identifier_mismatch",
                               "detail": f"h_only={only_h}, d_only={only_d}"})
        del h, d, merged
        gc.collect()

joins_df = pd.DataFrame(join_rows)
print("Household/derived join checks:", len(joins_df))
if not joins_df.empty:
    display(joins_df.head(12))

Household/derived join checks: 505


,wave,country,imputation,matched_keys,h_only_keys,d_only_keys
0,2010,AT,1,2380,0,0
1,2010,BE,1,2327,0,0
2,2010,CY,1,1237,0,0
3,2010,DE,1,3565,0,0
4,2010,E1,1,6197,0,0
5,2010,ES,1,6106,0,0
6,2010,FI,1,10989,0,0
7,2010,FR,1,15006,0,0
8,2010,GR,1,2971,0,0
9,2010,IT,1,7951,0,0


## 5. Produce the CSV diagnostics

Each CSV opens in Excel. `02_variable_missingness_by_imp.csv` retains imputation-specific information; the summary gives an across-imputation *range*, **not** a pooled household sample. No values are cleaned, recoded or removed.

In [5]:
# Summarise country-wave-variable missingness across the five imputations.
if missing_df.empty:
    missing_summary_df = pd.DataFrame()
else:
    missing_summary_df = (
        missing_df.groupby(["wave", "country", "table", "variable"], as_index=False)
        .agg(imputations_available=("imputation", "nunique"),
             rows_per_imputation_min=("rows", "min"),
             rows_per_imputation_max=("rows", "max"),
             mean_missing_pct=("missing_pct", "mean"),
             min_missing_pct=("missing_pct", "min"),
             max_missing_pct=("missing_pct", "max"),
             min_observed_n=("observed_n", "min"),
             max_observed_n=("observed_n", "max"),
             any_extended_stata_missing=("stata_extended_missing_n", lambda x: bool(x.gt(0).any())),
             any_empty_strings=("empty_string_n", lambda x: bool(x.gt(0).any())))
    )
    for col in ("mean_missing_pct", "min_missing_pct", "max_missing_pct"):
        missing_summary_df[col] = missing_summary_df[col].round(3)

if schema_df.empty:
    presence_df = pd.DataFrame()
else:
    raw_schema = schema_df.loc[~schema_df["stata_missing_companion"]].copy()
    presence_df = (
        raw_schema.groupby(["wave", "table", "variable"], as_index=False)
        .agg(imputations_available=("imputation", "nunique"),
             parquet_dtypes=("parquet_type", lambda s: "; ".join(sorted(set(s)))))
    )

report_frames = {
    "00_input_file_inventory.csv": files_df,
    "01_country_wave_counts.csv": samples_df,
    "02_variable_missingness_by_imp.csv": missing_df,
    "03_variable_missingness_summary.csv": missing_summary_df,
    "04_numeric_category_frequencies.csv": pd.DataFrame(
        code_rows, columns=["wave", "country", "table", "variable", "raw_numeric_code", "count", "pct_of_nonmissing"]),
    "05_stata_extended_missing_codes.csv": pd.DataFrame(
        extended_rows, columns=["wave", "country", "table", "imputation", "variable", "stata_missing_code", "count"]),
    "06_h_d_identifier_matches.csv": joins_df,
    "07_issues_to_review.csv": pd.DataFrame(
        issues, columns=["wave", "country", "table", "imputation", "variable", "issue", "detail"]),
    "08_variable_presence.csv": presence_df,
    "09_text_encoding_flags.csv": pd.DataFrame(
        text_rows, columns=["wave", "country", "table", "imputation", "variable",
                            "unicode_replacement_character_rows", "unicode_control_character_rows"]),
}

for filename, report in report_frames.items():
    report.to_csv(OUTPUT / filename, index=False, encoding="utf-8-sig")
    print(f"{filename:<46} {len(report):>10,} rows")

(OUTPUT / "README.txt").write_text(
    "HFCS selected-raw diagnostic exports\n"
    "Inputs: sibling HFCS Data Download/HFCS_selected_raw/ (outside git).\n"
    "No source observations or variables have been changed.\n"
    "02: each country, wave, table, imputation and variable separately.\n"
    "03: descriptive mean/min/max of missing percentages across the five imputations.\n"
    "04: integer-coded HH/DH categorical distributions from imputation 1 only; "
    "distributions containing cells smaller than 5 are omitted. Codes are unlabeled.\n"
    "05: original Stata extended-missing codes from __stata_missing companions; "
    "counts may include small cells, review before sharing.\n"
    "06: distinct household-key coverage in H versus D.\n"
    "07: structural/data issues to review, not automatically repaired.\n"
    "09: suspicious decoded-text counts only, no text contents.\n"
    "Neither code values -1/-2 nor blank strings are automatically treated as missing.\n"
    "IMPORTANT: These reports contain restricted-data aggregates; REVIEW ALL SMALL CELLS "
    "before pushing to a public GitHub repository.\n",
    encoding="utf-8",
)

print("\nReports saved to:", OUTPUT)
if not missing_summary_df.empty:
    display(missing_summary_df.sort_values("mean_missing_pct", ascending=False).head(15))

00_input_file_inventory.csv                            50 rows
01_country_wave_counts.csv                          1,010 rows
02_variable_missingness_by_imp.csv                100,960 rows
03_variable_missingness_summary.csv                20,192 rows
04_numeric_category_frequencies.csv                 7,830 rows
05_stata_extended_missing_codes.csv                42,346 rows
06_h_d_identifier_matches.csv                         505 rows
07_issues_to_review.csv                                 0 rows
08_variable_presence.csv                              996 rows
09_text_encoding_flags.csv                              0 rows

Reports saved to: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\Research-Assistant-Git\HFCS\diagnostics


,wave,country,table,variable,imputations_available,rows_per_imputation_min,rows_per_imputation_max,mean_missing_pct,min_missing_pct,max_missing_pct,min_observed_n,max_observed_n,any_extended_stata_missing,any_empty_strings
19985,2023,SI,h,sa0110,5,1987,1987,100.0,100.0,100.0,0,0,True,False
16043,2023,AT,d,dl1120c,5,2849,2849,100.0,100.0,100.0,0,0,True,False
10270,2017,MT,d,dhageh1,5,1004,1004,100.0,100.0,100.0,0,0,True,False
4098,2014,ES,d,dl1110c,5,6120,6120,100.0,100.0,100.0,0,0,True,False
10347,2017,MT,d,dl1120c,5,1004,1004,100.0,100.0,100.0,0,0,True,False
4071,2014,ES,d,di1520,5,6120,6120,100.0,100.0,100.0,0,0,True,False
6171,2014,PL,d,da21012,5,3455,3455,100.0,100.0,100.0,0,0,True,False
6121,2014,NL,h,hh0203,5,1284,1284,100.0,100.0,100.0,0,0,True,False
13469,2021,GR,h,sa0110,5,3386,3386,100.0,100.0,100.0,0,0,True,False
19783,2023,PT,h,sa0210,5,5465,5465,100.0,100.0,100.0,0,0,True,False


## How to interpret the output

Start with `00_input_file_inventory.csv` and `07_issues_to_review.csv`, then `01_country_wave_counts.csv` and `03_variable_missingness_summary.csv`.

- A **missing rate** here is based on actual nulls plus preserved Stata extended missing codes. A negative numeric value is **not automatically missing**.
- `empty_string_n` counts blank/whitespace strings, **separately** from nulls.
- Categorical frequencies show *raw numeric codes*, **not decoded survey answer labels**. Tiny-cell category distributions are omitted.
- `unique_observed_n` counts raw non-null values in the Parquet column and can include ordinary numeric values but not extended missing-code companions.
- The notebook does **not** calculate wealth moments, pool imputations statistically, or merge H and D into a new dataset.
- No underlying microdata are written into the repository. **Review all diagnostic CSVs for disclosure risk before pushing them**.